# Ship Coating Quality Data Labeling Visualization

Visualize segmentation and bounding box labeling of defect images.

In [ ]:
import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from pathlib import Path
import random

# For Korean text display (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'  # For Mac
plt.rcParams['axes.unicode_minus'] = False

## 1. Data Path Configuration

In [ ]:
# Data paths (English directory structure)
IMAGE_DIR = 'Dataset/01.images'
LABEL_DIR = 'Dataset/02.labels'

# Category definitions
CATEGORIES = {
    101: 'Normal',
    201: 'Water Spotting',
    202: 'Sagging',
    203: 'Coating Separation',
    204: 'Pinhole',
    205: 'Crack',
    206: 'Blister',
    207: 'Foreign Material',
    301: 'Welding Damage',
    302: 'Scratch',
    303: 'Peeling'
}

CATEGORIES_KR = {
    101: '정상',
    201: '워터스포팅',
    202: '흐름',
    203: '도막분리',
    204: '핀홀',
    205: '균열',
    206: '부풀음',
    207: '이물질포함',
    301: '용접손상',
    302: '스크래치',
    303: '도막떨어짐'
}

# Colors for each category (BGR format)
COLORS = {
    101: (0, 255, 0),      # Normal - Green
    201: (255, 0, 0),      # Water Spotting - Blue
    202: (0, 165, 255),    # Sagging - Orange
    203: (0, 255, 255),    # Coating Separation - Yellow
    204: (255, 0, 255),    # Pinhole - Magenta
    205: (0, 0, 255),      # Crack - Red
    206: (255, 255, 0),    # Blister - Cyan
    207: (128, 0, 128),    # Foreign Material - Purple
    301: (255, 165, 0),    # Welding Damage - Sky
    302: (0, 128, 255),    # Scratch - Orange-Red
    303: (128, 128, 128)   # Peeling - Gray
}

# English folder name mapping
CATEGORY_DIRS = {
    'coating_damage': 'Coating Damage',
    'painting_defect': 'Painting Defect',
    'normal': 'Normal'
}

SUBCATEGORY_DIRS = {
    'peeling': 'Peeling',
    'scratch': 'Scratch',
    'welding_damage': 'Welding Damage',
    'crack': 'Crack',
    'coating_separation': 'Coating Separation',
    'blister': 'Blister',
    'water_spotting': 'Water Spotting',
    'foreign_material': 'Foreign Material',
    'pinhole': 'Pinhole',
    'sagging': 'Sagging',
    'deck': 'Deck',
    'engine_room': 'Engine Room',
    'stern': 'Stern',
    'bow': 'Bow',
    'cabin': 'Cabin',
    'engine_cover': 'Engine Cover',
    'outer_plate': 'Outer Plate',
    'tank': 'Tank',
    'pipe': 'Pipe',
    'hatch_cover': 'Hatch Cover'
}

## 2. Data Loading and Visualization Functions

In [ ]:
def load_json(json_path):
    """Load JSON file"""
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def visualize_annotation(image_path, json_path, show_segmentation=True, show_bbox=True, use_korean=True):
    """
    Visualize annotations on image
    
    Args:
        image_path: Image file path
        json_path: JSON annotation file path
        show_segmentation: Whether to show segmentation polygons
        show_bbox: Whether to show bounding boxes
        use_korean: Use Korean labels (True) or English (False)
    """
    # Check image path
    if not os.path.exists(image_path):
        print(f"[ERROR] Image file not found: {image_path}")
        return
    
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"[ERROR] Cannot load image: {image_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Load JSON
    data = load_json(json_path)
    
    # Prepare visualization
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=16)
    axes[0].axis('off')
    
    # Annotation overlay image
    overlay = img.copy()
    
    # Select category dict
    cat_dict = CATEGORIES_KR if use_korean else CATEGORIES
    
    # Process each annotation
    for ann in data['annotations']:
        category_id = ann['category_id']
        category_name = cat_dict.get(category_id, 'Unknown')
        color = COLORS.get(category_id, (255, 255, 255))
        
        # Draw Segmentation
        if show_segmentation and 'segmentation' in ann:
            seg = ann['segmentation']
            # Reshape polygon points
            points = np.array(seg).reshape(-1, 2).astype(np.int32)
            
            # Semi-transparent polygon overlay
            overlay_temp = overlay.copy()
            cv2.fillPoly(overlay_temp, [points], color)
            cv2.addWeighted(overlay_temp, 0.4, overlay, 0.6, 0, overlay)
            
            # Polygon border
            cv2.polylines(overlay, [points], True, color, 3)
        
        # Draw Bounding Box
        if show_bbox and 'bbox' in ann:
            x, y, w, h = [int(v) for v in ann['bbox']]
            cv2.rectangle(overlay, (x, y), (x+w, y+h), color, 3)
            
            # Label text
            label = f"{category_name}"
            if 'attributes' in ann and 'part' in ann['attributes']:
                label += f" ({ann['attributes']['part']})"
            
            # Text background
            (text_width, text_height), _ = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2
            )
            cv2.rectangle(overlay, (x, y-text_height-10), 
                         (x+text_width, y), color, -1)
            cv2.putText(overlay, label, (x, y-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # Display annotation overlay image
    axes[1].imshow(overlay)
    axes[1].set_title('Labeling Visualization', fontsize=16)
    axes[1].axis('off')
    
    # Display metadata
    info_text = f"File: {data['images'][0]['file_name']}\n"
    info_text += f"Resolution: {data['images'][0]['width']} x {data['images'][0]['height']}\n"
    info_text += f"Annotations: {len(data['annotations'])}\n"
    
    for i, ann in enumerate(data['annotations'], 1):
        cat_name = cat_dict.get(ann['category_id'], 'Unknown')
        info_text += f"  #{i} {cat_name}"
        if 'area' in ann:
            info_text += f" (Area: {ann['area']:.0f}px²)"
        info_text += "\n"
    
    plt.figtext(0.5, 0.01, info_text, ha='center', fontsize=10, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

## 3. Dataset Exploration

In [ ]:
def get_all_samples():
    """Collect all sample file paths"""
    samples = []
    
    for category in ['coating_damage', 'painting_defect', 'normal']:
        label_category_dir = os.path.join(LABEL_DIR, category)
        
        if not os.path.exists(label_category_dir):
            continue
            
        for subcategory in os.listdir(label_category_dir):
            subcategory_dir = os.path.join(label_category_dir, subcategory)
            
            if not os.path.isdir(subcategory_dir):
                continue
            
            for json_file in os.listdir(subcategory_dir):
                if not json_file.endswith('.json'):
                    continue
                
                json_path = os.path.join(subcategory_dir, json_file)
                
                # Find image file
                image_filename = json_file.replace('.json', '')
                # Try JPG or jpg extension
                image_path_jpg = os.path.join(IMAGE_DIR, category, subcategory, image_filename + '.JPG')
                image_path_jpg_lower = os.path.join(IMAGE_DIR, category, subcategory, image_filename + '.jpg')
                
                if os.path.exists(image_path_jpg):
                    image_path = image_path_jpg
                elif os.path.exists(image_path_jpg_lower):
                    image_path = image_path_jpg_lower
                else:
                    continue
                
                samples.append({
                    'category': category,
                    'category_display': CATEGORY_DIRS.get(category, category),
                    'subcategory': subcategory,
                    'subcategory_display': SUBCATEGORY_DIRS.get(subcategory, subcategory),
                    'image_path': image_path,
                    'json_path': json_path
                })
    
    return samples

# Collect all samples
all_samples = get_all_samples()
print(f"Total samples: {len(all_samples)}")

# Category statistics
from collections import Counter
category_counts = Counter([s['category_display'] for s in all_samples])
subcategory_counts = Counter([f"{s['category_display']}/{s['subcategory_display']}" for s in all_samples])

print("\n=== Category Distribution ===")
for cat, count in category_counts.items():
    print(f"{cat}: {count} samples")

print("\n=== Subcategory Distribution ===")
for subcat, count in sorted(subcategory_counts.items()):
    print(f"{subcat}: {count} samples")

## 4. Coating Damage Samples Visualization

In [ ]:
# Filter coating damage samples
damage_samples = [s for s in all_samples if s['category'] == 'coating_damage']

print(f"Coating Damage samples: {len(damage_samples)}")
print("Subtypes:", set([s['subcategory_display'] for s in damage_samples]))

In [ ]:
# Peeling sample visualization
peeling_samples = [s for s in damage_samples if s['subcategory'] == 'peeling']
if peeling_samples:
    sample = random.choice(peeling_samples)
    print(f"Peeling Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Scratch sample visualization
scratch_samples = [s for s in damage_samples if s['subcategory'] == 'scratch']
if scratch_samples:
    sample = random.choice(scratch_samples)
    print(f"Scratch Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Welding damage sample visualization
welding_samples = [s for s in damage_samples if s['subcategory'] == 'welding_damage']
if welding_samples:
    sample = random.choice(welding_samples)
    print(f"Welding Damage Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

## 5. Painting Defect Samples Visualization

In [ ]:
# Filter painting defect samples
defect_samples = [s for s in all_samples if s['category'] == 'painting_defect']

print(f"Painting Defect samples: {len(defect_samples)}")
print("Subtypes:", set([s['subcategory_display'] for s in defect_samples]))

In [ ]:
# Crack sample visualization (may have multiple annotations)
crack_samples = [s for s in defect_samples if s['subcategory'] == 'crack']
if crack_samples:
    sample = random.choice(crack_samples)
    print(f"Crack Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Coating separation sample visualization
separation_samples = [s for s in defect_samples if s['subcategory'] == 'coating_separation']
if separation_samples:
    sample = random.choice(separation_samples)
    print(f"Coating Separation Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Blister sample visualization
blister_samples = [s for s in defect_samples if s['subcategory'] == 'blister']
if blister_samples:
    sample = random.choice(blister_samples)
    print(f"Blister Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Water spotting sample visualization
water_samples = [s for s in defect_samples if s['subcategory'] == 'water_spotting']
if water_samples:
    sample = random.choice(water_samples)
    print(f"Water Spotting Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Foreign material sample visualization
foreign_samples = [s for s in defect_samples if s['subcategory'] == 'foreign_material']
if foreign_samples:
    sample = random.choice(foreign_samples)
    print(f"Foreign Material Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Pinhole sample visualization
pinhole_samples = [s for s in defect_samples if s['subcategory'] == 'pinhole']
if pinhole_samples:
    sample = random.choice(pinhole_samples)
    print(f"Pinhole Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# Sagging sample visualization
sagging_samples = [s for s in defect_samples if s['subcategory'] == 'sagging']
if sagging_samples:
    sample = random.choice(sagging_samples)
    print(f"Sagging Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

## 6. Normal Samples Verification

In [ ]:
# Normal samples (no annotations)
normal_samples = [s for s in all_samples if s['category'] == 'normal']
if normal_samples:
    sample = random.choice(normal_samples)
    print(f"Normal Sample Visualization")
    visualize_annotation(sample['image_path'], sample['json_path'])

## 7. Annotation Statistics Analysis

In [ ]:
# Collect annotation statistics
annotation_stats = []

for sample in all_samples:
    data = load_json(sample['json_path'])
    
    for ann in data['annotations']:
        stat = {
            'category': sample['category_display'],
            'subcategory': sample['subcategory_display'],
            'category_id': ann['category_id'],
            'category_name': CATEGORIES.get(ann['category_id'], 'Unknown'),
            'has_segmentation': 'segmentation' in ann,
            'has_bbox': 'bbox' in ann,
            'area': ann.get('area', 0),
            'part': ann.get('attributes', {}).get('part', 'Unknown'),
            'quality': ann.get('attributes', {}).get('quality', 'Unknown')
        }
        annotation_stats.append(stat)

import pandas as pd
df_stats = pd.DataFrame(annotation_stats)

print("=== Annotation Statistics ===")
print(f"\nTotal annotations: {len(df_stats)}")
print(f"\nWith segmentation: {df_stats['has_segmentation'].sum()}")
print(f"With bounding box: {df_stats['has_bbox'].sum()}")

print("\n=== Annotations by Category ===")
print(df_stats['category_name'].value_counts())

print("\n=== Annotations by Quality ===")
print(df_stats['quality'].value_counts())

print("\n=== Annotations by Ship Part ===")
print(df_stats['part'].value_counts())

In [ ]:
# Defect area distribution
defect_areas = df_stats[df_stats['area'] > 0]['area']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(defect_areas, bins=30, edgecolor='black')
plt.xlabel('Area (px²)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Defect Area Distribution', fontsize=14)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(defect_areas)
plt.ylabel('Area (px²)', fontsize=12)
plt.title('Defect Area Box Plot', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean defect area: {defect_areas.mean():.2f} px²")
print(f"Median: {defect_areas.median():.2f} px²")
print(f"Min: {defect_areas.min():.2f} px²")
print(f"Max: {defect_areas.max():.2f} px²")

## 8. Sample Explorer

In [ ]:
# Select desired category and subcategory
selected_category = 'painting_defect'  # 'coating_damage', 'painting_defect', 'normal'
selected_subcategory = 'crack'  # Check subcategory names from distribution above

filtered = [s for s in all_samples 
            if s['category'] == selected_category 
            and s['subcategory'] == selected_subcategory]

print(f"{CATEGORY_DIRS.get(selected_category, selected_category)}/"
      f"{SUBCATEGORY_DIRS.get(selected_subcategory, selected_subcategory)} samples: {len(filtered)}")

# Visualize 3 random samples
for sample in random.sample(filtered, min(3, len(filtered))):
    visualize_annotation(sample['image_path'], sample['json_path'])